# 🤖 Week 1: What Is an AI Agent?
### AgentEdu · Free Agentic AI Course · agentedu.org

**Created by Jothsna Praveena Pendyala** — AI Platform Architect & Senior Data Scientist, Infosys / AT&T

---

**In this notebook you will:**
- Understand the difference between a tool and an agent
- Trace the Think → Act → Observe loop in real Python code
- Build your first simple rule-based agent from scratch
- Complete an interactive activity: the Agent Hunt

**No AI API key required. No installs. Just run each cell in order.**

> 💡 New to notebooks? Click each cell and press **Shift + Enter** to run it.

---
## Part 1: Tools vs. Agents

A **tool** does exactly one thing when you tell it to.
An **agent** pursues a **goal** by taking a sequence of actions — and keeps going until it's done.

Let's see this in code.

In [ ]:
# ── A TOOL: does one thing, stops ──────────────────────────────
def calculator_tool(expression):
    """A tool: give it an input, get one output, done."""
    return eval(expression)

result = calculator_tool("12 * 8 + 5")
print(f"Calculator result: {result}")
print("The tool answered. Now it stops. It will not do anything else.")

In [ ]:
# ── AN AGENT: pursues a goal across multiple steps ─────────────
# This is a simple simulation. A real agent uses an LLM to think.
# Here, we simulate the reasoning to make the loop visible.

def simple_research_agent(goal, knowledge_base):
    """
    A minimal agent that searches a knowledge base to answer a question.
    Demonstrates: multi-step reasoning, tool use, and stopping when done.
    """
    print(f"🎯 GOAL: {goal}")
    print("-" * 50)

    memory = []     # what the agent has learned so far
    step = 0
    max_steps = 6

    # Define the agent's tools
    def search_tool(query):
        """Search our knowledge base for relevant facts."""
        return [item for item in knowledge_base if query.lower() in item.lower()]

    def summarize_tool(facts):
        """Combine facts into a summary."""
        return f"Summary: Found {len(facts)} relevant facts: " + " | ".join(facts[:3])

    # Agent reasoning rules (simplified — a real agent uses an LLM here)
    search_terms = goal.lower().split()[:2]  # extract key words from goal
    facts_found = []

    while step < max_steps:
        step += 1
        print(f"\n--- Step {step} ---")

        # ── THINK ────────────────────────────────────────────────
        if len(facts_found) < 2:
            thought = f"I need more facts. I'll search for '{search_terms[0]}'"
        elif len(facts_found) < 4:
            thought = f"I have {len(facts_found)} facts. Let me search more broadly."
        else:
            thought = "I have enough facts. Time to summarize and finish."

        print(f"💭 THINK: {thought}")

        # ── ACT ──────────────────────────────────────────────────
        if "summarize" in thought:
            action = "summarize_tool"
            result = summarize_tool(facts_found)
            print(f"⚡  ACT: Called {action}")
        else:
            term = search_terms[0] if step % 2 == 1 else search_terms[-1]
            action = f"search_tool('{term}')"
            result = search_tool(term)
            print(f"⚡  ACT: Called {action}")

        # ── OBSERVE ──────────────────────────────────────────────
        if isinstance(result, list):
            new_facts = [r for r in result if r not in facts_found]
            facts_found.extend(new_facts)
            observation = f"Found {len(new_facts)} new facts. Total so far: {len(facts_found)}"
        else:
            observation = f"Generated summary: {result[:80]}..."
            print(f"👁  OBSERVE: {observation}")
            print("\n✅ GOAL REACHED. Agent stopping.")
            print(f"\n{result}")
            break

        print(f"👁  OBSERVE: {observation}")
        memory.append({"step": step, "action": action, "observation": observation})

    return memory


# Knowledge base (imagine this is the internet)
KB = [
    "AI agents use the Think-Act-Observe loop to reason toward goals",
    "Agents can use tools like web search, calculators, and databases",
    "A single agent handles simple tasks; multi-agent systems tackle complex ones",
    "LLMs power agent reasoning by predicting what to do next",
    "Memory allows agents to track what they've done across multiple steps",
    "RAG gives agents access to real information beyond their training data",
    "Agent safety is critical — hallucination and bias are real risks",
    "The AI engineer role grew 143% year-over-year in demand in 2025",
]

memory = simple_research_agent(
    goal="Tell me about AI agents and their tools",
    knowledge_base=KB
)

---
## ✏️ Activity 1: Trace the Loop

Look at the output above and answer these questions:

In [ ]:
# Fill in your answers below (replace the '...' strings)

your_answers = {
    "How many steps did the agent take?": "...",   # look at the output above
    "What tools did the agent use?": "...",         # look for 'ACT' lines
    "How did the agent know when to stop?": "...",  # what triggered the break?
    "What would happen if the knowledge base was empty?": "...",  # think it through
}

print("Your answers:")
for q, a in your_answers.items():
    status = "✓" if a != "..." else "○"
    print(f"  {status} {q}")
    if a != "...":
        print(f"      → {a}")

answered = sum(1 for a in your_answers.values() if a != "...")
print(f"\nCompleted {answered}/{len(your_answers)} questions")

---
## Part 2: Build Your Own Simple Agent

Now you'll build a simple agent from scratch. This one decides what to do based on rules — no AI involved yet. That comes in Week 2.

Your agent's job: answer trivia questions using a knowledge base.

In [ ]:
# ── BUILD YOUR TRIVIA AGENT ────────────────────────────────────

class TriviaAgent:
    """
    A simple rule-based agent that answers trivia questions.
    This is your first agent. It's not AI-powered — it uses rules.
    But it follows the same Think → Act → Observe loop.
    """

    def __init__(self, name):
        self.name = name
        self.memory = []           # what this agent has learned
        self.confidence = 0        # how confident it is in its answers
        self.knowledge_base = {
            "capital of france": "Paris",
            "largest planet": "Jupiter",
            "speed of light": "299,792,458 meters per second",
            "inventor of telephone": "Alexander Graham Bell",
            "year python created": "1991",
        }

    def think(self, question):
        """Reason about the question. Return a plan."""
        q_lower = question.lower().strip("?")
        # Check if we've answered this before
        if q_lower in [m["question"] for m in self.memory]:
            return "already_answered"
        # Check if we know the answer
        for key in self.knowledge_base:
            if key in q_lower:
                return f"search_kb:{key}"
        return "unknown"

    def act(self, plan, question):
        """Execute the plan. Return a result."""
        if plan == "already_answered":
            prev = next(m for m in self.memory if m["question"] == question.lower().strip("?"))
            return f"I already answered this: {prev['answer']}"
        elif plan.startswith("search_kb:"):
            key = plan.split(":")[1]
            return self.knowledge_base[key]
        else:
            return "I don't know the answer to that question."

    def observe(self, question, answer):
        """Store what happened in memory."""
        self.memory.append({"question": question.lower().strip("?"), "answer": answer})
        if "don't know" not in answer and "already" not in answer:
            self.confidence += 1

    def answer(self, question):
        """Run one full Think → Act → Observe cycle."""
        print(f"\n❓ Question: {question}")
        plan   = self.think(question);   print(f"   💭 Think:   {plan}")
        result = self.act(plan, question);print(f"   ⚡  Act:    {result}")
        self.observe(question, result);  print(f"   👁  Observe: Stored in memory (total: {len(self.memory)})")
        return result


# Create your agent
agent = TriviaAgent(name="TriviaBot")

# Ask it some questions
questions = [
    "What is the capital of France?",
    "What is the largest planet?",
    "Who created Python?",           # not in knowledge base — watch what happens
    "What is the capital of France?", # asking again — watch the memory kick in
]

print(f"Agent '{agent.name}' is running...")
for q in questions:
    agent.answer(q)

print(f"\n📊 Final stats: {agent.confidence} confident answers, {len(agent.memory)} total questions answered")

---
## ✏️ Activity 2: The Agent Hunt (Notebook Version)

Choose a technology you use. Fill in its agent profile below.

In [ ]:
# ── THE AGENT HUNT ─────────────────────────────────────────────
# Fill in each field. Replace the placeholder strings.

my_agent = {
    "product":    "Spotify",                             # ← Change this to your chosen product
    "goal":       "Keep me listening as long as possible",
    "THINK":      "What songs has this user skipped, saved, or replayed?",
    "ACT":        "Curate and play a playlist",
    "OBSERVE":    "Track which songs get skipped and which get added to library",
    "autonomous": "Mostly autonomous — but I can skip or save songs to guide it",
    "failure":    "It keeps recommending music from artists I listened to years ago",
}

# ── Print your profile ─────────────────────────────────────────
print("=" * 55)
print(f"  AGENT PROFILE: {my_agent['product']}")
print("=" * 55)
print(f"  Goal:       {my_agent['goal']}")
print(f"  💭 Think:   {my_agent['THINK']}")
print(f"  ⚡  Act:    {my_agent['ACT']}")
print(f"  👁  Observe: {my_agent['OBSERVE']}")
print(f"  Autonomy:   {my_agent['autonomous']}")
print(f"  Failure:    {my_agent['failure']}")
print("=" * 55)
print()
print("✅ Now change 'product' above to YOUR chosen technology")
print("   and re-run this cell with Shift+Enter.")

---
## ✏️ Challenge: Improve Your Agent

The `TriviaAgent` above is limited. It can only answer questions in its knowledge base.

**Your challenge:** Add 3 new facts to the knowledge base and test them.

In [ ]:
# ── YOUR TURN: Add facts and test ─────────────────────────────

# Create a new agent with an extended knowledge base
agent2 = TriviaAgent(name="MyAgent")

# Add 3 new facts (replace these with your own!)
agent2.knowledge_base["largest ocean"] = "Pacific Ocean"
agent2.knowledge_base["year agentic ai emerged"] = "2023-2024"
agent2.knowledge_base["most spoken language"] = "Mandarin Chinese"

# Test your new facts
test_questions = [
    "What is the largest ocean?",
    "When did agentic AI emerge?",
    "What is the most spoken language in the world?",
    "What is the meaning of life?",   # not in KB — what happens?
]

print(f"Testing '{agent2.name}' with {len(agent2.knowledge_base)} facts in knowledge base...")
for q in test_questions:
    agent2.answer(q)

print(f"\nYour agent answered {agent2.confidence} questions correctly.")
print("\n💡 Notice: The agent doesn't pretend to know things it doesn't.")
print("   That's a design choice. Real AI agents often get this wrong — they hallucinate.")
print("   We'll cover hallucination in Week 2.")

---
## Summary

In this notebook, you:

- ✅ Saw the **difference between a tool and an agent** in real code
- ✅ Traced the **Think → Act → Observe loop** step by step
- ✅ Built your first **rule-based agent** from scratch
- ✅ Completed the **Agent Hunt** activity in Python
- ✅ Extended your agent's knowledge and tested it

---

## What's Next — Week 2: The Brain Behind the Agent

Right now your agent makes decisions using fixed rules you wrote. In Week 2, we replace those rules with an **LLM** — a Large Language Model — which can reason about any question, not just ones you pre-programmed.

We'll cover:
- How LLMs work (next-token prediction, explained simply)
- Why they hallucinate — and why that's a structural problem, not a bug
- How to write prompts that actually work
- The Prompt Engineering Challenge

---

**AgentEdu** · agentedu.org · Created by Jothsna Praveena Pendyala  
Free for any student, anywhere · CC BY 4.0  
📸 @jothsna.aitales · 📧 hello@agentedu.org